# Semana 1 — Scikit-Learn Pipelines e o Accuracy Paradox
**Fase 1: Math Foundations e ML Core** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/01-Phase-1-Math-and-ML-Core/01-sklearn-pipelines/notebooks/semana_01_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-01-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/01-Phase-1-Math-and-ML-Core/01-sklearn-pipelines/docs/Semana-01-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

## Passo 1 — Importações

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, accuracy_score, recall_score, precision_score)

## Passo 2 — Imbalanced data
Simulamos 100.000 transações com cerca de 0,5% de fraudes. O formato imita o dataset *Credit Card Fraud Detection* do Kaggle sem exigir download.

In [ ]:
X, y = make_classification(n_samples=100_000, n_features=20, n_informative=6, n_redundant=4,
                           weights=[0.995], class_sep=1.3, flip_y=0.001, random_state=42)
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(20)])
df["is_fraud"] = y
print(df["is_fraud"].value_counts(normalize=True).mul(100).round(2))

## Passo 3 — O baseline que não faz nada
Antes de treinar qualquer coisa, calcule a accuracy de um "modelo" que sempre responde 0 (não é fraude). Guarde esse número: ele é a régua.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_ingenuo = np.zeros_like(y_test)
acc_ingenuo = accuracy_score(y_test, y_ingenuo)
rec_ingenuo = recall_score(y_test, y_ingenuo)
print(f"Acurácia do modelo ingênuo: {acc_ingenuo:.4f} | Recall de fraude: {rec_ingenuo:.2f}")

## Passo 4 — Pipeline
Monte um `Pipeline` com duas etapas: `scaler` (`StandardScaler`) e `classifier` (`RandomForestClassifier` com `n_estimators=100, random_state=42, n_jobs=-1`). Treine com uma linha.

In [ ]:
pipeline_base = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
])
pipeline_base.fit(X_train, y_train)

## Passo 5 — O accuracy paradox

In [ ]:
def avaliar(modelo, nome, limiar=0.5):
    proba = modelo.predict_proba(X_test)[:, 1]
    y_pred = (proba >= limiar).astype(int)
    print(f"=== {nome} (limiar={limiar}) ===")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Fraude"], digits=3, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["Normal", "Fraude"]).plot(cmap="Blues")
    plt.title(nome); plt.show()
    return proba

proba_base = avaliar(pipeline_base, "Pipeline base")

Compare a accuracy acima com a do modelo ingênuo. Quantas fraudes passaram (false negatives)?

## Passo 6 — A visão do arquiteto: `class_weight`
Crie `pipeline_balanceado`, idêntico ao anterior, com `class_weight="balanced_subsample"` e `min_samples_leaf=5` na floresta. Treine e avalie.

In [ ]:
pipeline_balanceado = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                          class_weight="balanced_subsample", min_samples_leaf=5)),
])
pipeline_balanceado.fit(X_train, y_train)
proba_bal = avaliar(pipeline_balanceado, "Pipeline balanceado")

## Passo 7 — O threshold como alavanca de negócio
Use `precision_recall_curve` nas probabilidades do pipeline balanceado. Depois encontre o **maior threshold** que ainda garante Recall de fraude ≥ 0,80 e reavalie com ele.

In [ ]:
prec, rec, lim = precision_recall_curve(y_test, proba_bal)
plt.plot(rec, prec); plt.xlabel("Recall"); plt.ylabel("Precisão"); plt.title("Curva Precisão-Recall"); plt.grid(alpha=.3); plt.show()
ok = rec[:-1] >= 0.80            # rec/prec têm um elemento a mais que lim
limiar_escolhido = lim[ok].max()
print("Limiar escolhido:", round(float(limiar_escolhido), 3))
_ = avaliar(pipeline_balanceado, "Balanceado com limiar ajustado", limiar=round(float(limiar_escolhido), 3))

## Passo 8 — Do erro ao custo
Suponha que cada fraude não detectada custa R$ 2.000 e cada false positive custa R$ 15 (análise manual). Calcule o custo total no test set para vários thresholds e ache o que minimiza o custo.

In [ ]:
def custo(limiar, proba, c_fn=2000, c_fp=15):
    y_pred = (proba >= limiar).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, y_pred).ravel()
    return fn * c_fn + fp * c_fp

grade = np.linspace(0.01, 0.9, 60)
custos = [custo(t, proba_bal) for t in grade]
melhor = grade[int(np.argmin(custos))]
plt.plot(grade, custos); plt.axvline(melhor, ls="--"); plt.xlabel("Limiar"); plt.ylabel("Custo (R$)"); plt.show()
print(f"Limiar de menor custo: {melhor:.2f} | custo: R$ {min(custos):,.0f}")

## Fechamento
Escreva em `docs/impacto-de-negocio.md` (3 a 5 frases): qual threshold você recomendaria, quanto custa cada tipo de erro e como essa mesma análise se aplicaria a um hallucination detector em um sistema RAG.